Importation des bibliothèques :

In [3]:
# Manipulation de données
import json
import os
import re
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import requests

# Clients optionnels : le cours reste exécutable sans eux
try:
    from dotenv import load_dotenv
    load_dotenv()      # lit le fichier .env s'il existe
except ImportError:
    print("Paquet absent : python-dotenv (le fichier .env ne sera pas lu)")

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

try:
    import tiktoken
except ImportError:
    tiktoken = None

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

print("✓ Bibliothèques importées avec succès !")
print(f"  numpy    : {np.__version__}")
print(f"  pandas   : {pd.__version__}")
print(f"  openai   : {'disponible' if OpenAI else 'absent (pip install openai)'}")
print(f"  tiktoken : {'disponible' if tiktoken else 'absent (pip install tiktoken)'}")

✓ Bibliothèques importées avec succès !
  numpy    : 2.5.2
  pandas   : 3.0.5
  openai   : disponible
  tiktoken : disponible


Liste des modèles disponibles sous OLLAMA, + passage de la variable pour choisir le modèle OLLAMA

In [4]:
OLLAMA_URL = os.getenv("OLLAMA_URL") or "http://localhost:11434"
MODELE_LOCAL = os.getenv("OLLAMA_MODEL") or "qwen3:8b"


def etat_ollama() -> list[str]:
    """Affiche la version d'Ollama et rend la liste des modèles installés."""
    try:
        version = requests.get(f"{OLLAMA_URL}/api/version", timeout=5).json()["version"]
        modeles = [m["name"] for m in requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()["models"]]
    except (requests.RequestException, KeyError, ValueError):
        print("Ollama ne répond pas : lancez l'application, puis relancez cette cellule.")
        return []
    print(f"Ollama {version} : {len(modeles)} modèle(s) installé(s) : {modeles}")
    return modeles


modeles_installes = etat_ollama()
OLLAMA_PRET = MODELE_LOCAL in modeles_installes
if not OLLAMA_PRET:
    print(f"Modèle {MODELE_LOCAL} indisponible : les cellules qui en ont besoin seront ignorées.")

Ollama 0.40.14 : 4 modèle(s) installé(s) : ['qwen3-embedding:0.6b', 'qwen3:4b-instruct', 'qwen3:4b', 'qwen3:8b']


Mise en place du code de config, si aucun env n'est présent il passera sur ollama directement 

In [5]:
def prix(nom: str, defaut: float = 0.0) -> float:
    """Lit un prix dans l'environnement ; une variable vide vaut le défaut."""
    return float(os.getenv(nom) or defaut)


@dataclass
class ConfigLLM:
    fournisseur: str
    base_url: str
    api_key: str
    model: str
    prix_entree: float = 0.0   # $ par million de tokens
    prix_sortie: float = 0.0
    prix_cache: float = 0.0
    effort: str | None = None  # réglage du raisonnement, si le modèle en a un


def config_depuis_env() -> ConfigLLM:
    if (os.getenv("LLM_PROVIDER") or "ollama") == "ollama":
        return ConfigLLM("ollama", f"{OLLAMA_URL}/v1", "ollama", MODELE_LOCAL,
                         effort=os.getenv("LLM_EFFORT") or "none")   # qwen3:8b respecte ce réglage, qwen3:4b non (section 2.4)
    return ConfigLLM("api", os.environ["LLM_BASE_URL"], os.environ["LLM_API_KEY"], os.environ["LLM_MODEL"],
                     prix("LLM_PRIX_ENTREE"), prix("LLM_PRIX_SORTIE"),
                     prix("LLM_PRIX_CACHE", prix("LLM_PRIX_ENTREE")), os.getenv("LLM_EFFORT") or None)


cfg = config_depuis_env()
print(f"Fournisseur : {cfg.fournisseur} | modèle : {cfg.model} | prix : {cfg.prix_entree} / {cfg.prix_sortie} $")

Fournisseur : ollama | modèle : qwen3:8b | prix : 0.0 / 0.0 $


fonction d'appel

In [6]:
def cout_appel(entree: int, sortie: int, p_entree: float, p_sortie: float,
               cache: int = 0, p_cache: float = 0.0) -> float:
    """Coût en dollars ; les prix sont exprimés par million de tokens."""
    return ((entree - cache) * p_entree + cache * p_cache + sortie * p_sortie) / 1_000_000


@dataclass
class Reponse:
    texte: str
    modele: str
    entree: int
    cache: int
    sortie: int
    cout: float
    fin: str
    latence: float = 0.0


def creer_client(cfg: ConfigLLM):
    if OpenAI is None:
        raise RuntimeError("Paquet absent : pip install openai")
    return OpenAI(base_url=cfg.base_url, api_key=cfg.api_key)


def appeler(messages: list[dict], cfg: ConfigLLM | None = None, client=None,
            max_tokens: int = 512, temperature: float | None = None, afficher: bool = True) -> Reponse:
    cfg = cfg or config_depuis_env()
    client = client or creer_client(cfg)
    options = {}
    if cfg.effort:
        options["reasoning_effort"] = cfg.effort
    if temperature is not None:
        options["temperature"] = temperature

    r = client.chat.completions.create(model=cfg.model, messages=messages, max_tokens=max_tokens, **options)

    u = r.usage
    details = getattr(u, "prompt_tokens_details", None)      # absent chez certains serveurs
    cache = getattr(details, "cached_tokens", 0) or 0
    cout = cout_appel(u.prompt_tokens, u.completion_tokens, cfg.prix_entree, cfg.prix_sortie, cache, cfg.prix_cache)
    rep = Reponse(r.choices[0].message.content or "", cfg.model, u.prompt_tokens, cache,
                  u.completion_tokens, cout, r.choices[0].finish_reason or "")
    if afficher:
        print(f"[{rep.modele}] entrée={rep.entree} (cache {rep.cache}) sortie={rep.sortie} coût≈{rep.cout:.6f} $")
    if rep.fin == "length":
        print("⚠️ Réponse coupée par max_tokens : si elle est vide, le raisonnement a tout consommé.")
    return rep

Test modèle : 

In [7]:
def modele_pret(cfg: ConfigLLM | None = None) -> bool:
    """Le modèle configuré est-il utilisable ? Dit quoi faire si ce n'est pas le cas."""
    cfg = cfg or config_depuis_env()
    if OpenAI is None:
        print("Non testé : paquet openai absent (pip install openai).")
        return False
    if cfg.fournisseur == "ollama" and cfg.model not in etat_ollama():
        print(f"Non testé : modèle local absent (ollama pull {cfg.model}).")
        return False
    return True


MODELE_PRET = modele_pret()
CLIENT = creer_client(cfg) if MODELE_PRET else None

if MODELE_PRET:
    rep = appeler([{"role": "user", "content": "Explique ce qu'est un token, en une phrase."}], max_tokens=200)
    print(rep.texte)

Ollama 0.40.14 : 4 modèle(s) installé(s) : ['qwen3-embedding:0.6b', 'qwen3:4b-instruct', 'qwen3:4b', 'qwen3:8b']
[qwen3:8b] entrée=28 (cache 0) sortie=43 coût≈0.000000 $
Un token est une unité de base dans un système de blockchain ou une chaîne de caractères utilisée pour représenter un actif, une autorité ou une action dans un protocole.


#### Fonction search_product

In [ ]:
def search_product(name: String, code: int, limite: int)
    